# 07. Face Embedding - arcface_mobilefacenet

| 항목 | 내용 |
|---|---|
| 대상 모델 | `arcface_mobilefacenet.dxnn` |
| 카테고리 | **Face Embedding** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 512개 숫자로 된 출력이 이름표가 아니라 좌표라는 사실을 두 벡터의 비교 방식으로 설명할 수 있다.
2. L2 정규화를 거친 벡터에서 코사인 유사도가 내적과 같아지는 이유를 계산으로 보일 수 있다.
3. 임계값을 넘는지로 같은 사람 여부를 판정하는 절차를 실행해 확인할 수 있다.
4. 학습에 없던 사람도 판정할 수 있는 구조와 분류 모델의 차이를 설명할 수 있다.

## 1. Face Embedding (얼굴 임베딩)

**얼굴 한 장을 정해진 길이의 숫자 목록으로 바꾸어 다른 얼굴과 견줄 수 있게 만드는 작업**

같은 사람의 얼굴은 가까운 자리에, 다른 사람의 얼굴은 먼 자리에 놓이도록 학습된다. 그래서 두 사람이 같은
사람인지 판정하는 일이 **두 숫자 목록 사이의 거리를 재는 일**로 바뀐다.

**출력의 성격**

이름표가 아니라 좌표다. 한 장만으로는 아무 뜻이 없고 **둘을 비교해야** 뜻이 생긴다. 학습에 없던 사람도
좌표는 나오므로, 얼굴을 새로 등록할 때 모델을 다시 학습하지 않아도 된다.

**활용 사례**

- 등록된 얼굴과의 대조로 출입 확인
- 사진 묶음에서 같은 인물 모으기
- 명단에 있는 사람인지 확인

**포함되지 않는 작업**

- 사진에서 얼굴 찾기 (Face Detection)
- 나이·표정 같은 속성 판정 (Face Attribute)
- 정해진 사람 목록 중 하나로 분류 (Image Classification)

## 2. 전체 파이프라인

```text
sample/img/face_pair/  정렬된 파일 3장을 차례로
  → 전처리         BGR→RGB · 112×112 직접 축소 · uint8 유지
  → run()          한 장마다 입력 Tensor 하나를 NPU 에 전달
  → Public Output  452  [1, 512]  float32
  → 후처리         flatten → L2 정규화 → EmbeddingResult
  → 비교           첫 장을 기준으로 두고 나머지와 코사인 유사도 계산
  → 시각화         기준과 현재를 좌우로 붙이고 유사도·판정을 아래 띠에 적음
```

![07. Face Embedding 전체 파이프라인](resources/07_Face_Embedding_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `data` |
| Shape / Dtype | `[1, 112, 112, 3]` / `uint8` |
| 채널 순서 | NHWC |
| Color / 값 범위 | RGB / 0~255 |
| Resize 방식 | 직접 축소 (비율 유지 없음) |
| Normalization | 없음 |

**입력이 사진 한 장이 아니라 디렉토리다**

- **정렬된 파일명 순서로 읽는다** — `1_reference.jpg` · `2_same.jpg` · `3_different.jpg`. 첫 장이 기준이 되므로
  파일명 앞의 번호가 순서를 정한다
- **얼굴이 이미 잘려 정렬돼 있어야 한다** — 이 모델에는 얼굴을 찾는 단계가 없다. 눈·코 위치를 맞춰 자른 얼굴을
  전제로 학습됐으므로 전신 사진을 넣으면 좌표가 흐트러진다
- **사진 입력만 받는다** — 영상·카메라 입력은 실행 진입점에서 거부된다. 비교할 기준이 필요한 작업이라 단일
  모델 예제에 그 구조가 없다

### 3.2 출력

| Tensor | Shape / dtype | 의미 | Public |
|---|---|---|---|
| `452` | `[1, 512]` / `float32` | 얼굴 한 장의 512차원 좌표 | O |

`Public` 은 `InferenceEngine.run()` 이 돌려주는 값인지를 가리키는 표시다. 이 모델은 Task 가 `npu_0` 하나뿐이라
NPU 가 계산한 값이 그대로 나온다.

**shape 각 자리의 뜻**

| index | 값 | 뜻 |
|---|---:|---|
| 0 | 1 | 한 번에 처리하는 얼굴 장수 |
| 1 | 512 | 좌표의 차원 수. 각 자리가 축 하나 |

`float32` 하나가 4바이트이므로 `512` 개는 2,048 바이트다. **512개 숫자 각각에는 사람이 읽을 뜻이 없다.**
3번 자리가 눈 크기라거나 100번 자리가 코 길이라고 해석할 근거가 없으며, 뜻은 **여러 벡터를 함께 놓고 볼 때만**
생긴다.

**비교는 방향으로 한다.** 두 벡터가 얼마나 같은 방향을 보는지를 코사인 유사도로 재며, 1 에 가까우면 같은
방향이고 0 에 가까우면 무관하다.

```text
cos(a, b) = (a · b) / (|a| × |b|)
```

**후처리가 L2 정규화를 하는 이유가 여기 있다.** 길이를 1 로 맞춰 두면 분모가 `1 × 1` 이 되어 코사인 유사도가
**내적 한 번**으로 끝난다. 얼굴 수천 장과 대조해야 하는 자리에서 나눗셈을 매번 하지 않아도 되므로 계산이
가벼워진다.

길이를 1 로 맞추는 계산은 각 값을 벡터의 길이로 나누는 것이다. 이번 실행에서 기준 얼굴의 원시 벡터 길이는
`7.0254` 였고, 첫 값 `0.0785` 는 `0.0785 / 7.0254 = 0.0112` 가 되었다.

**판정 임계값은 0.4 다.** 유사도가 이 값을 넘으면 같은 사람으로 본다. 임계값은 모델 파일이 아니라 판정하는
쪽이 정하는 값이며, 낮추면 같은 사람을 놓치지 않지만 남을 통과시키고 높이면 그 반대가 된다.

### 3.3 Config.json

| key | 현재 값 | 기본값 | 의미 |
|---|---:|---:|---|
| `top_k` | 5 | 5 | 이 모델의 후처리가 읽지 않는 key |

**이 파일의 `top_k` 는 쓰이지 않는다.** 임베딩 후처리는 `normalize` 하나만 설정에서 찾고, 그 key 는 파일에
없으므로 코드 기본값 `True` 가 적용된다. `top_k` 는 상위 몇 개를 남길지를 정하는 값이라 분류 계열 후처리의
것이며, 예제 폴더를 같은 형태로 맞추다 남은 것으로 판단된다. 지워도 이 모델의 동작은 달라지지 않는다.

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. 입력이 112×112,
출력이 512 차원 하나인지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "arcface_mobilefacenet.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 디렉토리를 정하고 설정 파일을 읽는다. 이후 셀은 여기서 만든 이름을 그대로 쓴다.

In [ ]:
import json
import sys
import time
from dataclasses import dataclass, field

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

sys.dont_write_bytecode = True
IMAGE_DIR = DX_APP / "sample" / "img/face_pair"
CONFIG_PATH = DX_APP / "src" / "python_example" / "embedding" / "arcface_mobilefacenet" / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.is_file() else {}

IMAGE_PATHS = sorted(IMAGE_DIR.glob("*.jpg"))

print("config", CONFIG)
print("images", [path.name for path in IMAGE_PATHS])

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 이 모델의 후처리는 되돌릴
  좌표가 없어 받기만 하고 읽지 않는다
- `EmbeddingResult` — 후처리가 만들어 비교 단계로 넘기는 결과. 정규화된 벡터와 모델 종류를 담는다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale_x: float = 0.0
    scale_y: float = 0.0
    pad_x: int = 0
    pad_y: int = 0


@dataclass
class EmbeddingResult:
    """정규화된 얼굴 벡터. 원본 대응: common/base/i_processor.py"""

    embedding: np.ndarray = field(default=None)
    model_type: str = ""


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. `parse_model` 출력과 같아야 한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output:", output_info)

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
EMBED_DIM = output_info[0]["shape"][1]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT} / embedding dim = {EMBED_DIM}")

### 4.2 전처리

얼굴 사진 세 장을 각각 RGB 112×112 `uint8` 로 바꾼다. 세 장 모두 같은 전처리를 거쳐야 벡터끼리 비교할 수
있으며, 한 장만 다르게 처리하면 좌표가 어긋나 유사도가 틀린다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 112×112
- dtype — `uint8` 유지
- 세 장에 같은 절차 — 순서와 방식이 동일한지

#### 입력 이미지 확인

세 장을 읽어 나란히 표시한다. 파일명이 순서를 정하고 첫 장이 기준이 된다.

In [ ]:
images = [cv2.imread(str(path)) for path in IMAGE_PATHS]
for path, img in zip(IMAGE_PATHS, images):
    print(f"{path.name:18s} shape={img.shape} dtype={img.dtype}")

preview_height = 260
previews = [cv2.resize(img, (int(img.shape[1] * preview_height / img.shape[0]), preview_height))
            for img in images]
notebook_show_bgr(cv2.hconcat(previews), "1_reference | 2_same | 3_different", figsize=(11, 5))

#### 전처리 함수 정의

- `arcface_preprocessor_create` — 모델 입력 크기를 state 에 담음
- `arcface_preprocessor_process` — 얼굴 한 장을 입력 Tensor 와 Context 로 변환

In [ ]:
def arcface_preprocessor_create(input_width: int, input_height: int) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height}


def arcface_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]
    ctx.scale_x = float(ctx.input_width) / float(ctx.original_width)
    ctx.scale_y = float(ctx.input_height) / float(ctx.original_height)

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (ctx.input_width, ctx.input_height),
                         interpolation=cv2.INTER_LINEAR)
    return resized, ctx

#### 전처리 실행

세 장을 같은 함수로 처리해 입력 Tensor 목록을 만든다. shape 와 dtype 이 셋 다 같은지 확인한다.

In [ ]:
preprocessor_state = arcface_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
tensors, contexts = [], []
for path, img in zip(IMAGE_PATHS, images):
    tensor, ctx = arcface_preprocessor_process(preprocessor_state, img)
    tensors.append(tensor)
    contexts.append(ctx)
    print(f"{path.name:18s} tensor={tensor.shape} {tensor.dtype} "
          f"scale_x={ctx.scale_x:.4f} scale_y={ctx.scale_y:.4f}")

### 4.3 추론

#### DXNN Inference 실행

세 장을 차례로 NPU 에 넘긴다. 한 번의 `run()` 은 한 장만 처리하므로 비교를 하려면 여러 번 호출해야 한다.
출력되는 시간은 각 1회 측정값이고 런타임과의 왕복 비용이 포함돼 있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
raw_outputs = []
for path, tensor in zip(IMAGE_PATHS, tensors):
    started = time.perf_counter()
    outputs = engine.run([tensor])
    elapsed_ms = (time.perf_counter() - started) * 1000.0
    array = np.asarray(outputs[0])
    raw_outputs.append(outputs)
    print(f"{path.name:18s} {elapsed_ms:6.2f} ms  shape={array.shape} "
          f"min={array.min():7.4f} max={array.max():7.4f}")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- 길이 — 정규화 전 벡터의 길이가 1 이 아니다
- 값의 부호 — 음수가 섞여 있어 확률이 아니다
- 세 벡터의 길이 차이 — 사진마다 다르므로 나눗셈이 필요하다

In [ ]:
for path, outputs in zip(IMAGE_PATHS, raw_outputs):
    raw = np.asarray(outputs[0]).flatten()
    print(f"{path.name:18s} L2={np.linalg.norm(raw):7.4f} "
          f"min={raw.min():7.4f} max={raw.max():7.4f} 음수={int((raw < 0).sum())}/{raw.size}")

print("\nreference 앞 5개 값:", np.array2string(
    np.asarray(raw_outputs[0][0]).flatten()[:5], precision=4))

### 4.4 후처리

벡터를 1차원으로 펴고 길이 1 로 맞춘다. 순서는 flatten → L2 정규화이며, 정규화를 끄면 코사인 유사도를 계산할
때마다 두 벡터의 길이로 나누어야 한다.

**정규화는 설정에서 켜고 끌 수 있고 기본값이 켜짐이다.** 이 모델의 설정 파일에는 그 key 가 없어 기본값이
그대로 쓰인다.

#### 후처리 함수 정의

- `arcface_postprocessor_create` — 정규화 여부를 state 에 담음
- `arcface_postprocessor_process` — flatten 과 L2 정규화를 거쳐 Result 생성
- `arcface_cosine_similarity` — 두 벡터의 코사인 유사도

In [ ]:
def arcface_postprocessor_create(config: dict) -> dict:
    """Config 와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {"normalize": config.get("normalize", True)}


def arcface_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    embedding = np.asarray(outputs[0]).flatten().astype(np.float32)
    if state["normalize"]:
        norm = np.linalg.norm(embedding)
        if norm > 0:
            embedding = embedding / norm
    return [EmbeddingResult(embedding=embedding, model_type="face_embedding")]


def arcface_cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """두 벡터가 같은 방향을 보는 정도. 원본 대응: EmbeddingVisualizer._cosine_similarity()"""
    norm_a, norm_b = np.linalg.norm(a), np.linalg.norm(b)
    if norm_a < 1e-8 or norm_b < 1e-8:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))

#### 후처리 실행

확인할 것:

- 정규화 후 길이 — 셋 다 `1.000000` 인지
- 내적과 코사인 유사도 — 정규화 뒤에는 두 값이 같은지
- 유사도와 임계값 — 같은 사람 쪽이 0.4 를 넘고 다른 사람 쪽이 못 넘는지

**`1_reference.jpg` 와 `2_same.jpg` 는 파일 내용이 같다.** 두 벡터가 완전히 일치해 유사도가 정확히 1 이 되므로,
이 값은 "같은 사람의 다른 사진" 이 아니라 **같은 입력에 대한 상한**으로 읽어야 한다. 같은 사람의 다른 사진을
넣었을 때의 값은 이 노트북 마지막의 `같은 사람의 다른 사진으로 다시 확인` 에서 직접 잰다.

In [ ]:
postprocessor_state = arcface_postprocessor_create(CONFIG)
embeddings = []
for path, outputs in zip(IMAGE_PATHS, raw_outputs):
    result = arcface_postprocessor_process(postprocessor_state, outputs, contexts[0])[0]
    embeddings.append(result.embedding)
    print(f"{path.name:18s} dim={result.embedding.size} "
          f"L2={np.linalg.norm(result.embedding):.6f} model_type={result.model_type!r}")

print("\n정규화 뒤에는 내적이 코사인 유사도와 같다:")
print(f"  내적       = {float(np.dot(embeddings[0], embeddings[2])):.4f}")
print(f"  코사인     = {arcface_cosine_similarity(embeddings[0], embeddings[2]):.4f}")

print("\n기준(1_reference) 과의 유사도:")
THRESHOLD = 0.4
for path, embedding in zip(IMAGE_PATHS[1:], embeddings[1:]):
    similarity = arcface_cosine_similarity(embeddings[0], embedding)
    verdict = "SAME" if similarity > THRESHOLD else "DIFFERENT"
    print(f"  {path.name:18s} {similarity:.4f} → {verdict} (임계값 {THRESHOLD})")

### 4.5 시각화

#### Visualizer 코드

기준 얼굴을 안에 담아 두고, 다음 얼굴이 올 때마다 좌우로 붙인 그림과 유사도·판정을 만든다. 첫 호출은 기준을
담기만 하고 그림을 내지 않으므로 세 장을 넣으면 그림은 두 장 나온다. 그림에 넣는 글자는 영문만 쓴다 —
OpenCV 기본 폰트에 한글 자모가 없어 한글을 넣으면 글자가 깨진다.

In [ ]:
class ArcfaceVisualizer:
    """기준 얼굴을 담아 두고 다음 얼굴과 좌우 비교 그림을 만든다. 원본 대응: Runner.visualize()"""

    def __init__(self, threshold: float = 0.4):
        self.threshold = threshold
        self._ref_image = None
        self._ref_embedding = None

    def visualize(self, image: np.ndarray, results: list):
        embedding = np.asarray(results[0].embedding, dtype=np.float32)
        if self._ref_image is None:
            self._ref_image = image.copy()
            self._ref_embedding = embedding
            return None

        similarity = arcface_cosine_similarity(self._ref_embedding, embedding)
        return self._canvas(self._ref_image, image, similarity)

    @staticmethod
    def _fit(image: np.ndarray, box_w: int, box_h: int) -> np.ndarray:
        height, width = image.shape[:2]
        scale = min(box_w / width, box_h / height)
        return cv2.resize(image, (int(width * scale), int(height * scale)))

    def _canvas(self, ref: np.ndarray, cur: np.ndarray, similarity: float) -> np.ndarray:
        canvas_h, canvas_w, bar_h, gap = max(cur.shape[0], 640), max(cur.shape[1], 960), 60, 6
        image_h, half_w = canvas_h - bar_h, (canvas_w - gap) // 2
        canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)

        for index, source in enumerate((ref, cur)):
            fitted = self._fit(source, half_w, image_h)
            height, width = fitted.shape[:2]
            x = index * (half_w + gap) + (half_w - width) // 2
            y = (image_h - height) // 2
            canvas[y:y + height, x:x + width] = fitted
            cv2.putText(canvas, ("Reference", "Current")[index], (x + 4, y + 22),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2, cv2.LINE_AA)

        cv2.line(canvas, (half_w + gap // 2, 0), (half_w + gap // 2, image_h), (80, 80, 80), 1)

        overlay = canvas.copy()
        cv2.rectangle(overlay, (0, image_h), (canvas_w, canvas_h), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.7, canvas, 0.3, 0, canvas)

        matched = similarity > self.threshold
        color = (0, 200, 0) if matched else (0, 0, 220)
        text = f"Cosine Similarity: {similarity:.4f}  [{'SAME' if matched else 'DIFFERENT'}]"
        cv2.putText(canvas, text, (10, image_h + 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2,
                    cv2.LINE_AA)
        return canvas

#### 결과 시각화

세 장을 순서대로 넣어 비교 그림 두 장을 만든다. 첫 장은 기준으로만 쓰이므로 그림이 나오지 않는다.

In [ ]:
visualizer = ArcfaceVisualizer(threshold=THRESHOLD)
for path, img, embedding in zip(IMAGE_PATHS, images, embeddings):
    canvas = visualizer.visualize(img, [EmbeddingResult(embedding=embedding)])
    if canvas is None:
        print(f"{path.name:18s} 기준으로 저장 (그림 없음)")
        continue
    notebook_show_bgr(canvas, f"Reference vs {path.name}", figsize=(11, 7))

#### 같은 사람의 다른 사진으로 다시 확인

위 결과의 `1.0000` 은 같은 파일을 두 번 넣어 얻은 값이다. 같은 사람이지만 사진이 다를 때의 유사도를 보려면
기준 얼굴을 조금 바꿔 다시 넣으면 된다. 아래 셀은 기준 얼굴을 좌우로 뒤집고 밝기를 올린 뒤 같은 파이프라인을
통과시킨다. **원본 파일을 고치지 않고 메모리에서만 바꾼 검증용 입력**이다.

In [ ]:
verification_variant = cv2.convertScaleAbs(cv2.flip(images[0], 1), alpha=1.15, beta=12)
variant_tensor, variant_ctx = arcface_preprocessor_process(preprocessor_state, verification_variant)
variant_embedding = arcface_postprocessor_process(
    postprocessor_state, engine.run([variant_tensor]), variant_ctx)[0].embedding

variant_similarity = arcface_cosine_similarity(embeddings[0], variant_embedding)
print(f"기준 vs 변형본(좌우 반전 + 밝기 상승) = {variant_similarity:.4f} → "
      f"{'SAME' if variant_similarity > THRESHOLD else 'DIFFERENT'}")
print(f"기준 vs 다른 사람                     = "
      f"{arcface_cosine_similarity(embeddings[0], embeddings[2]):.4f}")

variant_visualizer = ArcfaceVisualizer(threshold=THRESHOLD)
variant_visualizer.visualize(images[0], [EmbeddingResult(embedding=embeddings[0])])
notebook_show_bgr(
    variant_visualizer.visualize(verification_variant,
                                 [EmbeddingResult(embedding=variant_embedding)]),
    "Reference vs modified same face", figsize=(11, 7))